# Session 2 — AI SQL
**Capital Management AI Workshop**

In this session you will use Cortex AI functions to:
- Extract structured fields from free-text analyst notes (`AI_EXTRACT`)
- Classify notes into event types (`AI_CLASSIFY`)
- Extract metadata from staged investment PDFs
- Build two materialized tables: `EXTRACTED_INVESTMENT_INSIGHTS` and `DOCUMENT_CHUNKS`

**Run each cell in order from top to bottom.**

In [ ]:
USE DATABASE SLC_PORTFOLIO_AI;
USE SCHEMA PORTFOLIO_ANALYTICS;
USE WAREHOUSE CAPITAL_WH;

---
## Step 1 — Preview Analyst Notes

Before running AI functions, inspect the raw data to understand the text format.

In [ ]:
SELECT
    note_id,
    ticker,
    analyst_name,
    note_date,
    LEFT(note_text, 200) AS note_preview
FROM ANALYST_NOTES
LIMIT 5;

---
## Step 2 — AI_EXTRACT: Structured Data from Analyst Notes

`AI_EXTRACT` reads free-text and returns structured JSON based on a `responseFormat` schema you define. Each key is a field name; the value is a description of what to extract.

Run on 5 sample notes first to verify extraction quality before scaling to the full table.

In [ ]:
SELECT
    note_id,
    LEFT(note_text, 80) AS note_preview,
    AI_EXTRACT(
        text => note_text,
        responseFormat => {
            'ticker_symbol':      'What stock ticker symbol is primarily referenced? (e.g. RY, NVDA)',
            'recommendation':     'What is the analyst recommendation? (Buy, Hold, Sell, Overweight, Underweight, Market Perform)',
            'price_target':       'What is the 12-month price target in dollars, as a number only?',
            'key_risk':           'What is the primary risk factor mentioned?',
            'investment_horizon': 'What is the investment time horizon? (short-term, medium-term, long-term)'
        }
    ) AS extracted
FROM ANALYST_NOTES
LIMIT 5;

---
## Step 3 — AI_CLASSIFY: Categorise Analyst Notes

`AI_CLASSIFY` assigns each note to one of your predefined categories and returns a `label` and a confidence `score` (0–1). No model selection needed.

In [ ]:
SELECT
    note_id,
    LEFT(note_text, 80) AS note_preview,
    AI_CLASSIFY(
        note_text,
        ['Earnings Update', 'Risk Flag', 'Price Target Revision',
         'Sector Commentary', 'Initiation of Coverage', 'Regulatory Alert']
    ) AS classification
FROM ANALYST_NOTES
LIMIT 10;

---
## Step 4 — AI_EXTRACT on Staged Investment Documents

`AI_EXTRACT` can work directly on files in a stage via `TO_FILE()` — no separate parse step needed.
This samples 3 documents to verify the extraction schema before the full pipeline.

In [ ]:
SELECT
    RELATIVE_PATH,
    AI_EXTRACT(
        file => TO_FILE('@INVESTMENT_DOCS', RELATIVE_PATH),
        responseFormat => {
            'document_type':      'What type of document is this? (Research Report, Fund Prospectus, Quarterly Report, Investment Memo, Market Commentary, Risk Assessment, Compliance Note)',
            'primary_security':   'What is the primary security, fund, or asset class discussed?',
            'key_recommendation': 'What is the key recommendation or conclusion?',
            'risk_level':         'What overall risk level is implied? (Low, Medium, High)',
            'time_horizon':       'What investment or reporting time horizon is referenced?'
        }
    ) AS extracted
FROM DIRECTORY('@INVESTMENT_DOCS')
LIMIT 3;

---
## Step 5 — Batch Extraction Pipeline: EXTRACTED_INVESTMENT_INSIGHTS

Run `AI_EXTRACT` and `AI_COMPLETE` across the full `ANALYST_NOTES` table and materialise the results.

- `AI_EXTRACT` pulls 5 structured fields from each note
- `AI_COMPLETE` generates a one-sentence portfolio action recommendation per note
- Materialising once means downstream queries are fast and AI costs are only incurred once

> **This cell runs AI functions across all rows — allow 1–2 minutes to complete.**

In [ ]:
CREATE OR REPLACE TABLE EXTRACTED_INVESTMENT_INSIGHTS AS
WITH extracted AS (
    SELECT
        NOTE_ID,
        SECURITY_ID,
        TICKER,
        ANALYST_NAME,
        NOTE_DATE,
        SENTIMENT,
        AI_EXTRACT(
            NOTE_TEXT,
            {
                'ticker_symbol':      'What stock ticker symbol is primarily referenced?',
                'recommendation':     'What is the analyst recommendation? (Buy, Hold, Sell, Overweight, Underweight, Market Perform)',
                'price_target':       'What is the 12-month price target in dollars, as a number only?',
                'key_risk':           'What is the primary risk factor mentioned?',
                'investment_horizon': 'What is the investment time horizon? (short-term, medium-term, long-term)'
            }
        ) AS result
    FROM ANALYST_NOTES
)
SELECT
    NOTE_ID,
    SECURITY_ID,
    TICKER,
    ANALYST_NAME,
    NOTE_DATE,
    SENTIMENT,
    result:response:ticker_symbol::VARCHAR      AS EXTRACTED_TICKER,
    result:response:recommendation::VARCHAR     AS EXTRACTED_RECOMMENDATION,
    result:response:price_target::VARCHAR       AS PRICE_TARGET,
    result:response:key_risk::VARCHAR           AS KEY_RISK,
    result:response:investment_horizon::VARCHAR AS INVESTMENT_HORIZON,
    AI_COMPLETE(
        'claude-sonnet-4-6',
        'You are a portfolio manager. Based on the recommendation and risk below, write exactly one actionable portfolio action sentence (max 20 words). Recommendation: '
        || result:response:recommendation::VARCHAR
        || '. Key risk: ' || result:response:key_risk::VARCHAR || '.'
    )::VARCHAR AS PORTFOLIO_ACTION
FROM extracted;

In [ ]:
-- Verify row count and inspect sample output
SELECT COUNT(*) AS total_rows FROM EXTRACTED_INVESTMENT_INSIGHTS;

In [ ]:
SELECT
    NOTE_ID,
    TICKER,
    EXTRACTED_RECOMMENDATION,
    PRICE_TARGET,
    KEY_RISK,
    INVESTMENT_HORIZON,
    LEFT(PORTFOLIO_ACTION, 100) AS PORTFOLIO_ACTION
FROM EXTRACTED_INVESTMENT_INSIGHTS
LIMIT 10;

---
## Step 6 — Parse a Single Investment PDF

Test `AI_PARSE_DOCUMENT` on one file before running the full pipeline.
`mode: LAYOUT` preserves document structure — headers, paragraphs, and tables.

In [ ]:
-- List files available in the stage
SELECT RELATIVE_PATH FROM DIRECTORY('@INVESTMENT_DOCS') LIMIT 5;

In [ ]:
-- Parse a single document to inspect the extracted text
SELECT
    RELATIVE_PATH,
    LEFT(
        AI_PARSE_DOCUMENT(
            TO_FILE('@INVESTMENT_DOCS', RELATIVE_PATH),
            {'mode': 'LAYOUT'}
        ):content::VARCHAR,
        500
    ) AS text_preview
FROM DIRECTORY('@INVESTMENT_DOCS')
LIMIT 1;

---
## Step 7 — Build DOCUMENT_CHUNKS for Cortex Search

This pipeline chains four AI operations to transform all 15 investment PDFs into searchable chunks:

| Step | Function | Purpose |
|------|----------|---------|
| 1 | `AI_PARSE_DOCUMENT` | Extract full text from each PDF |
| 2 | `AI_CLASSIFY` | Label document type (used as filter attribute in Cortex Search) |
| 3 | `SPLIT_TEXT_RECURSIVE_CHARACTER` | Split into ~500-token chunks with 50-token overlap |
| 4 | `AI_EXTRACT` | Tag each chunk with the primary security ticker |

> **This cell parses 15 PDFs and runs AI on every chunk — allow 2–3 minutes to complete.**

In [ ]:
CREATE OR REPLACE TABLE DOCUMENT_CHUNKS AS
WITH parsed AS (
    SELECT
        RELATIVE_PATH AS document_name,
        AI_PARSE_DOCUMENT(
            TO_FILE('@INVESTMENT_DOCS', RELATIVE_PATH),
            {'mode': 'LAYOUT'}
        ):content::VARCHAR AS full_text,
        AI_CLASSIFY(
            AI_PARSE_DOCUMENT(
                TO_FILE('@INVESTMENT_DOCS', RELATIVE_PATH),
                {'mode': 'LAYOUT'}
            ):content::VARCHAR,
            ['Research Report', 'Fund Prospectus', 'Quarterly Report',
             'Investment Memo', 'Market Commentary', 'Risk Assessment', 'Compliance Note']
        ):label::VARCHAR AS document_type
    FROM DIRECTORY('@INVESTMENT_DOCS')
),
chunks AS (
    SELECT
        document_name,
        document_type,
        chunk.value::VARCHAR AS chunk_text,
        chunk.index          AS chunk_index
    FROM parsed,
    LATERAL FLATTEN(
        input => SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
            full_text, 'none', 500, 50
        )
    ) AS chunk
)
SELECT
    UUID_STRING()  AS chunk_id,
    document_name,
    document_type,
    chunk_text,
    chunk_index,
    AI_EXTRACT(
        chunk_text,
        {'security_ticker': 'What stock ticker symbol is primarily referenced? Return null if none mentioned.'}
    ):response:security_ticker::VARCHAR AS security_ticker
FROM chunks
WHERE LENGTH(chunk_text) > 100;

In [ ]:
SELECT COUNT(*) AS total_chunks FROM DOCUMENT_CHUNKS;

In [ ]:
-- Chunk breakdown by document type
SELECT document_type, COUNT(*) AS chunk_count
FROM DOCUMENT_CHUNKS
GROUP BY document_type
ORDER BY chunk_count DESC;

In [ ]:
SELECT
    chunk_id,
    document_name,
    document_type,
    security_ticker,
    chunk_index,
    LEFT(chunk_text, 150) AS chunk_preview
FROM DOCUMENT_CHUNKS
LIMIT 5;

---
## Session 2 Complete

You have built:

| Object | Description |
|--------|-------------|
| `EXTRACTED_INVESTMENT_INSIGHTS` | Structured extraction from all analyst notes with portfolio action recommendations |
| `DOCUMENT_CHUNKS` | 15 investment PDFs parsed, classified, chunked, and ticker-tagged — ready for Cortex Search |

Return to the workshop guide and proceed to **Session 3 — Cortex Search**.